# Cascade JEPA on ModernBERT-large: how much is JEPA, how much is fine-tuning?

One arm per run (`ARM`); everything else is identical across arms: data, split, model, predictor, optimiser, schedule and the **same total number of epochs**.

| Arm | Encoder | Stage 1 | Decision head | Isolates |
|---|---|---|---|---|
| **A** cascade | fine-tuned | the real Q·S target | cos(ẑ₂, yes/no vectors) / τ | – |
| **B** fair control | fine-tuned | none | plain 2 logits | **A − B: JEPA** |
| **C** shuffled target | fine-tuned | Q·S of a random other question | cos(ẑ₂, yes/no vectors) / τ | **A − C: the Q·S content** |
| **D** frozen control | frozen | none | plain 2 logits | **B − D: fine-tuning** |

**Schedule (all arms):** `stage1_epochs` + `stage2_epochs` epochs. A and C train the stage-1 loss alone first, then stage 2 (KL + stage-1 loss). B and D train the decision (KL) for all epochs. Warm-up then linear decay over the whole run; the best decision epoch by validation KL is kept.

**Q·S (stage-1 target)** is made once by a **frozen** copy of ModernBERT-large at its starting weights: the question alone (q) attends over the state's word tokens (S, encoded alone), in a standardised space, with sharpened attention (×`qs_sharpness`), minus the state's plain average. The check from `noul_cascade_jepa.ipynb` runs first.

**Measures:** validation, our test split, and the 600 Typed Decisions NOUL questions (Signpost-D-4B: 0.783 accuracy). A speed check after `speed_check_steps` steps prints the projected time and cost.

## Settings

In [ ]:
import os


def setting(name, default):
    value = os.environ.get(name.upper())
    if value is None:
        return default
    if isinstance(default, list):
        kind = type(default[0]) if default else str
        return [kind(v.strip()) for v in value.split(",") if v.strip()]
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    return type(default)(value)


CONFIG = {
    "arm": setting("arm", "A"),
    "seed": setting("seed", 42),
    "model_name": setting("model_name", "answerdotai/ModernBERT-large"),
    "data_path": setting("data_path", "data/signpost_d_decisions/train.jsonl"),
    "data_url": setting("data_url", "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
                                    "decider-jepa/data/signpost_d_decisions/train.jsonl"),
    "max_sequence_length": setting("max_sequence_length", 1024),
    "split_seed": setting("split_seed", 42),
    "train_ratio": setting("train_ratio", 0.80),
    "validation_ratio": setting("validation_ratio", 0.10),
    "encode_batch": setting("encode_batch", 16),

    # Q·S target (the recipe that passed the check on ModernBERT-base)
    "qs_space": setting("qs_space", "standardised"),
    "qs_sharpness": setting("qs_sharpness", 16.0),
    "qs_residual": setting("qs_residual", True),
    "qs_gate_same_state_cos": setting("qs_gate_same_state_cos", 0.95),

    # choice targets
    "target_words": setting("target_words", ["yes", "no"]),
    "target_repeats": setting("target_repeats", 64),
    "temperature": setting("temperature", 0.1),

    # predictor and heads
    "pred_width": setting("pred_width", 384),
    "pred_layers": setting("pred_layers", 2),
    "pred_heads": setting("pred_heads", 6),
    "dropout": setting("dropout", 0.1),

    # training
    "batch_size": setting("batch_size", 8),
    "encoder_lr": setting("encoder_lr", 1e-5),
    "head_lr": setting("head_lr", 3e-4),
    "weight_decay": setting("weight_decay", 0.01),
    "warmup_fraction": setting("warmup_fraction", 0.1),
    "stage1_epochs": setting("stage1_epochs", 3),
    "stage2_epochs": setting("stage2_epochs", 5),
    "stage1_weight_in_stage2": setting("stage1_weight_in_stage2", 1.0),
    "grad_clip": setting("grad_clip", 1.0),
    "speed_check_steps": setting("speed_check_steps", 50),

    "td_dataset": setting("td_dataset", "LocalLLaMA/typed-decisions"),
    "td_file": setting("td_file", "all/test-00000-of-00001.parquet"),
    "out_dir": setting("out_dir", "cascade_large"),
}

ARMS = {
    "A": {"fine_tune": True, "stage1": "real", "head": "cosine"},
    "B": {"fine_tune": True, "stage1": None, "head": "logits"},
    "C": {"fine_tune": True, "stage1": "shuffled", "head": "cosine"},
    "D": {"fine_tune": False, "stage1": None, "head": "logits"},
}
ARM = ARMS[CONFIG["arm"]]

for k, v in CONFIG.items():
    print(f"{k:26s}: {v}")
print("arm:", CONFIG["arm"], ARM)

## Data (as in `noul_cascade_jepa.ipynb`)

In [ ]:
import json
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = torch.device("cuda")

DATA_PATH = Path(CONFIG["data_path"])
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CONFIG["data_url"], DATA_PATH)


def parsed(value):
    return json.loads(value) if isinstance(value, str) else value


def noul_samples(records):
    out = []
    for record in records:
        state_text = "State:\n" + str(record["state"])
        for qid, q in record["questions"].items():
            if str(q.get("type", "")).lower() != "noul":
                continue
            gold = {str(k): float(v) for k, v in record["gold"][qid]["probabilities"].items()}
            question_text = "Question:\n" + str(q.get("instructions"))
            out.append({"group": record["id"], "question_id": qid,
                        "state_text": state_text, "question_text": question_text,
                        "joint_text": state_text + "\n\n" + question_text,
                        "gold": [gold["true"], gold["false"]]})
    return out


records = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8") if line.strip()]
samples = noul_samples(records)

groups = pd.Series([s["group"] for s in samples]).drop_duplicates().to_numpy()
shuffled = np.random.default_rng(CONFIG["split_seed"]).permutation(groups)
n_train = int(len(groups) * CONFIG["train_ratio"])
n_val = int(len(groups) * CONFIG["validation_ratio"])
split_of = {g: "train" for g in shuffled[:n_train]}
split_of.update({g: "validation" for g in shuffled[n_train:n_train + n_val]})
split_of.update({g: "test" for g in shuffled[n_train + n_val:]})
splits = {name: [s for s in samples if split_of[s["group"]] == name] for name in ("train", "validation", "test")}

td_frame = pd.read_parquet(hf_hub_download(CONFIG["td_dataset"], CONFIG["td_file"], repo_type="dataset"))
td_records = [{"id": r["id"], "state": parsed(r["state"]), "questions": parsed(r["questions"]),
               "gold": parsed(r["gold"])} for r in td_frame.to_dict("records")]
splits["td_test"] = noul_samples(td_records)

GOLD = {name: torch.tensor([r["gold"] for r in rows]) for name, rows in splits.items()}
print({k: len(v) for k, v in splits.items()})

## Targets from a frozen ModernBERT-large (its starting weights)

- **Choice vectors:** `yes` × 64 and `no` × 64, mean over the word tokens.
- **Q·S** for every question, and the check that it carries the question (needed by A and C; computed for every arm so they all report stage-1 cosine).

In [ ]:
import math
import time

import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
frozen = AutoModel.from_pretrained(CONFIG["model_name"], dtype=torch.bfloat16).to(device).eval()
HIDDEN = frozen.config.hidden_size


@torch.no_grad()
def word_states(texts):
    out = [None] * len(texts)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for s in range(0, len(order), CONFIG["encode_batch"]):
        idx = order[s:s + CONFIG["encode_batch"]]
        batch = tokenizer([texts[i] for i in idx], padding=True, truncation=True,
                          max_length=CONFIG["max_sequence_length"], return_tensors="pt",
                          return_special_tokens_mask=True)
        special = batch.pop("special_tokens_mask").bool()
        states = frozen(**batch.to(device)).last_hidden_state.float().cpu()
        keep = batch["attention_mask"].bool().cpu() & ~special
        for row, i in enumerate(idx):
            out[i] = states[row][keep[row]]
    return out


t0 = time.perf_counter()
CHOICES = torch.stack([t.mean(0) for t in word_states(
    [" ".join([w] * CONFIG["target_repeats"]) for w in CONFIG["target_words"]])]).to(device)
c = F.cosine_similarity(CHOICES[0], CHOICES[1], dim=0).item()
print(f"choice vectors: cos {c:.4f}, highest gold probability "
      f"{1 / (1 + math.exp(-(1 - c) / CONFIG['temperature'])):.4f}")

q_vec, S_of = {}, {}
for name, rows in splits.items():
    q_vec[name] = torch.stack([t.mean(0) for t in word_states([r["question_text"] for r in rows])])
    unique_states = sorted({r["state_text"] for r in rows})
    states = dict(zip(unique_states, word_states(unique_states)))
    S_of[name] = [states[r["state_text"]] for r in rows]
print(f"frozen encodings in {time.perf_counter() - t0:.0f} s")

train_tokens = torch.cat(list({id(t): t for t in S_of["train"]}.values()))
TOKEN_MEAN, TOKEN_STD = train_tokens.mean(0), train_tokens.std(0).clamp_min(1e-6)
del train_tokens


def qs_target(q, S):
    if CONFIG["qs_space"] == "standardised":
        S = (S - TOKEN_MEAN) / TOKEN_STD
        q = (q - TOKEN_MEAN) / TOKEN_STD
    w = torch.softmax(CONFIG["qs_sharpness"] * (S @ q) / math.sqrt(S.shape[1]), dim=0)
    target = w @ S
    return target - S.mean(0) if CONFIG["qs_residual"] else target


QS = {name: torch.stack([qs_target(q_vec[name][i], S_of[name][i]) for i in range(len(S_of[name]))])
      for name in splits}

# the check: Q·S of two questions on the same state must differ
by_group = {}
for i, r in enumerate(splits["train"]):
    by_group.setdefault(r["group"], []).append(i)
same = [F.cosine_similarity(QS["train"][a], QS["train"][b], dim=0).item()
        for idx in by_group.values() for j, a in enumerate(idx) for b in idx[j + 1:]]
pairs = np.random.default_rng(0).integers(0, len(QS["train"]), size=(3000, 2))
other = [F.cosine_similarity(QS["train"][a], QS["train"][b], dim=0).item()
         for a, b in pairs if splits["train"][a]["group"] != splits["train"][b]["group"]]
print(f"Q·S check: same state {np.mean(same):.4f} | different states {np.mean(other):.4f} "
      f"(gate: same < {CONFIG['qs_gate_same_state_cos']})")
assert np.mean(same) < CONFIG["qs_gate_same_state_cos"], "Q·S does not carry the question on this model."

if ARM["stage1"] == "shuffled":                          # arm C: each question gets another question's Q·S
    perm = torch.from_numpy(np.random.default_rng(CONFIG["seed"]).permutation(len(QS["train"])))
    QS_TRAIN_TARGET = QS["train"][perm]
    print("arm C: training Q·S targets shuffled across questions")
else:
    QS_TRAIN_TARGET = QS["train"]

mean_qs = QS["train"].mean(0)
for name in ("validation", "td_test"):
    base = F.cosine_similarity(QS[name], mean_qs.unsqueeze(0), dim=1).mean().item()
    print(f"{name}: cos(Q·S, mean training Q·S) = {base:.4f}")

del frozen, q_vec, S_of
torch.cuda.empty_cache()

## The model

- **Encoder:** ModernBERT-large (fine-tuned in A, B, C; frozen in D), on question + state together; gradient checkpointing, bf16.
- **Predictor:** the encoder's token states → linear to `pred_width`, a learned query token, `pred_layers` transformer layers, the query token's output → ẑ₁ (same size as the encoder).
- **Decision head:** `cosine` (A, C): ẑ₁ → MLP → ẑ₂, scored cos(ẑ₂, yes/no vectors) / τ. `logits` (B, D): ẑ₁ → MLP → 2 logits.

In [ ]:
import copy

import torch.nn as nn


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(CONFIG["model_name"])
        if ARM["fine_tune"]:
            self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        else:
            self.encoder.requires_grad_(False)
        w = CONFIG["pred_width"]
        self.inp = nn.Linear(HIDDEN, w)
        self.query = nn.Parameter(torch.randn(1, 1, w) * 0.02)
        layer = nn.TransformerEncoderLayer(w, CONFIG["pred_heads"], 4 * w, CONFIG["dropout"],
                                           batch_first=True, norm_first=True, activation="gelu")
        self.layers = nn.TransformerEncoder(layer, CONFIG["pred_layers"])
        self.z1 = nn.Sequential(nn.LayerNorm(w), nn.Linear(w, HIDDEN))
        out = HIDDEN if ARM["head"] == "cosine" else 2
        self.head = nn.Sequential(nn.LayerNorm(HIDDEN), nn.Linear(HIDDEN, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, out))

    def forward(self, input_ids, attention_mask):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            if ARM["fine_tune"]:
                states = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            else:
                with torch.no_grad():
                    states = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x = self.inp(states.float())
        x = torch.cat([self.query.expand(len(x), -1, -1), x], dim=1)
        pad = torch.cat([torch.zeros(len(x), 1, dtype=torch.bool, device=x.device), attention_mask == 0], dim=1)
        z1 = self.z1(self.layers(x, src_key_padding_mask=pad)[:, 0])
        h = self.head(z1)
        if ARM["head"] == "cosine":
            logits = F.cosine_similarity(h.unsqueeze(1), CHOICES.unsqueeze(0), dim=-1) / CONFIG["temperature"]
        else:
            logits = h
        return z1, logits


def tokens_of(name, idx):
    batch = tokenizer([splits[name][i]["joint_text"] for i in idx], padding=True, truncation=True,
                      max_length=CONFIG["max_sequence_length"], return_tensors="pt")
    return batch["input_ids"].to(device), batch["attention_mask"].to(device)


def kl(logits, gold):
    return (gold * (gold.clamp_min(1e-12).log() - F.log_softmax(logits.float(), dim=1))).sum(1)


@torch.no_grad()
def evaluate(model, name, batch=16):
    model.eval()
    n = len(splits[name])
    order = sorted(range(n), key=lambda i: len(splits[name][i]["joint_text"]))
    z_cos, logits_all, idx_all = [], [], []
    for s in range(0, n, batch):
        idx = order[s:s + batch]
        z1, logits = model(*tokens_of(name, idx))
        z_cos.append(F.cosine_similarity(z1.float(), QS[name][idx].to(device), dim=1))
        logits_all.append(logits.float())
        idx_all += idx
    logits = torch.cat(logits_all)
    gold = GOLD[name][idx_all].to(device)
    p = F.softmax(logits, dim=1)
    return {"stage1_cos": torch.cat(z_cos).mean().item(), "kl": kl(logits, gold).mean().item(),
            "uniform_kl": kl(torch.zeros_like(gold), gold).mean().item(),
            "top_choice": (p.argmax(1) == gold.argmax(1)).float().mean().item(),
            "brier": ((p - gold) ** 2).sum(1).mean().item(), "p_true_std": p[:, 0].std().item()}

## Training, with a checkpoint after every epoch

A rerun with the same run name resumes from the last finished epoch.

In [ ]:
OUT_DIR = Path(CONFIG["out_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = OUT_DIR / f"arm_{CONFIG['arm']}_latest.pt"
BEST = OUT_DIR / f"arm_{CONFIG['arm']}_best.pt"

torch.manual_seed(CONFIG["seed"])
model = Model().to(device)

encoder_params = [p for p in model.encoder.parameters() if p.requires_grad]
head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
groups_ = [{"params": head_params, "lr": CONFIG["head_lr"]}]
if encoder_params:
    groups_.append({"params": encoder_params, "lr": CONFIG["encoder_lr"]})
opt = torch.optim.AdamW(groups_, weight_decay=CONFIG["weight_decay"])

n_train = len(splits["train"])
steps_per_epoch = math.ceil(n_train / CONFIG["batch_size"])
total_epochs = CONFIG["stage1_epochs"] + CONFIG["stage2_epochs"]
total_steps = total_epochs * steps_per_epoch
warmup = int(CONFIG["warmup_fraction"] * total_steps)
def lr_factor(step):
    if step < warmup:
        return (step + 1) / max(1, warmup)                                   # warm-up
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup))     # linear decay


sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)

state = {"epoch": 0, "best_kl": float("inf"), "best_epoch": 0, "history": []}
if CKPT.exists():
    saved = torch.load(CKPT, map_location="cpu", weights_only=False)
    model.load_state_dict(saved["model"]); opt.load_state_dict(saved["opt"]); sched.load_state_dict(saved["sched"])
    state = saved["state"]
    print(f"RESUMED after epoch {state['epoch']}")

print(f"arm {CONFIG['arm']}: {steps_per_epoch} steps per epoch, {total_epochs} epochs, "
      f"trainable {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.0f} M")

RATES = {"L4": 0.80, "A10": 1.10, "L40S": 1.95, "A100": 2.50, "H100": 3.95}
gpu_name = torch.cuda.get_device_name(0)
rate = next((v for k, v in RATES.items() if k in gpu_name), None)


def save(path, obj):
    tmp = path.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


for epoch in range(state["epoch"] + 1, total_epochs + 1):
    in_stage1 = ARM["stage1"] is not None and epoch <= CONFIG["stage1_epochs"]
    w_stage1 = 0.0 if ARM["stage1"] is None else (1.0 if in_stage1 else CONFIG["stage1_weight_in_stage2"])
    w_kl = 0.0 if in_stage1 else 1.0
    model.train()
    if not ARM["fine_tune"]:
        model.encoder.eval()
    perm = torch.randperm(n_train, generator=torch.Generator().manual_seed(CONFIG["seed"] + epoch)).tolist()
    total, t0 = 0.0, time.perf_counter()
    for step, s in enumerate(range(0, n_train, CONFIG["batch_size"]), start=1):
        idx = perm[s:s + CONFIG["batch_size"]]
        z1, logits = model(*tokens_of("train", idx))
        loss = w_kl * kl(logits, GOLD["train"][idx].to(device)).mean()
        if w_stage1:
            loss = loss + w_stage1 * (1 - F.cosine_similarity(z1.float(), QS_TRAIN_TARGET[idx].to(device), dim=1)).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        opt.step(); sched.step()
        total += loss.item() * len(idx)
        if epoch == state["epoch"] + 1 and step == CONFIG["speed_check_steps"]:
            per_step = (time.perf_counter() - t0) / step
            remaining_h = per_step * (total_steps - (epoch - 1) * steps_per_epoch - step) / 3600
            cost = f", about ${remaining_h * rate:.2f} at ${rate}/h" if rate else ""
            print(f"SPEED CHECK on {gpu_name}: {per_step:.2f} s/step -> {remaining_h * 60:.0f} min left{cost}")
    v = evaluate(model, "validation")
    stage = "stage 1" if in_stage1 else "decision"
    print(f"arm {CONFIG['arm']} epoch {epoch} ({stage}) | train loss {total / n_train:.4f} | "
          f"{(time.perf_counter() - t0) / 60:.1f} min | val stage-1 cos {v['stage1_cos']:.4f} | "
          f"val KL {v['kl']:.4f} (uniform {v['uniform_kl']:.4f}) | top {v['top_choice']:.3f} | "
          f"P(true) std {v['p_true_std']:.3f}")
    state["history"].append({"epoch": epoch, "stage": stage, "train_loss": total / n_train, **v})
    if not in_stage1 and v["kl"] < state["best_kl"]:
        state["best_kl"], state["best_epoch"] = v["kl"], epoch
        save(BEST, {k: t.cpu() for k, t in model.state_dict().items()})
    state["epoch"] = epoch
    save(CKPT, {"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "state": state})

print(f"best decision epoch: {state['best_epoch']} (validation KL {state['best_kl']:.4f})")

## Results (best decision epoch)

In [ ]:
model.load_state_dict(torch.load(BEST, map_location="cpu", weights_only=False))
results = {name: evaluate(model, name) for name in ("train", "validation", "test", "td_test")}

table = pd.DataFrame(results).T
table.insert(0, "arm", CONFIG["arm"])
pd.set_option("display.width", 250)
display(table.round(4))

td = results["td_test"]
print(f"\nARM {CONFIG['arm']} RESULT | validation top {results['validation']['top_choice']:.4f} KL "
      f"{results['validation']['kl']:.4f} | test top {results['test']['top_choice']:.4f} KL "
      f"{results['test']['kl']:.4f} | Typed Decisions NOUL accuracy {td['top_choice']:.4f} KL {td['kl']:.4f} "
      f"(uniform {td['uniform_kl']:.4f}; Signpost-D-4B 0.7833 / 0.0952)")

with open(OUT_DIR / f"arm_{CONFIG['arm']}_results.json", "w") as f:
    json.dump({"config": CONFIG, "arm": ARM, "best_epoch": state["best_epoch"], "results": results,
               "history": state["history"]}, f, indent=1)
for path in (CKPT,):
    path.unlink(missing_ok=True)          # the large optimiser checkpoint is no longer needed
print("saved", OUT_DIR)